# Pixel-Perfect Depth (gangweix)

Diffusion-based relative depth. Runs one image at a time.

**Runtime:** GPU (L4 or A100) — slowest model

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, random, time, gc, sys, json, importlib, subprocess
import numpy as np
import cv2
import torch
from PIL import Image

DATASET_IMAGES = '/content/drive/MyDrive/Corn Seed Dataset/test/images'
DATASET_LABELS = '/content/drive/MyDrive/Corn Seed Dataset/test/labels'
SAVE_DIR       = '/content/drive/MyDrive/Corn Seed Dataset/depth_comparison_outputs'
SAMPLE_FILE    = os.path.join(SAVE_DIR, 'sample_images.txt')

assert os.path.isdir(DATASET_IMAGES), f'Dataset not found: {DATASET_IMAGES}'
os.makedirs(SAVE_DIR, exist_ok=True)

def save_depth(depth_np, stem, model_name):
    d = np.array(depth_np, dtype=np.float32)
    while d.ndim > 2: d = d[0]
    d_norm = (d - d.min()) / (d.max() - d.min() + 1e-8)
    for sub, img in [('depth', (d_norm*65535).astype(np.uint16)),
                     ('vis',   cv2.applyColorMap((d_norm*255).astype(np.uint8), cv2.COLORMAP_INFERNO))]:
        p = os.path.join(SAVE_DIR, model_name, sub)
        os.makedirs(p, exist_ok=True)
        cv2.imwrite(os.path.join(p, f'{stem}.png'), img)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('Setup done.')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Load shared sample list from Drive (run depth_pro.ipynb first)
assert os.path.exists(SAMPLE_FILE), f'Run depth_pro.ipynb first to create sample_images.txt'
SAMPLES = [l.strip() for l in open(SAMPLE_FILE) if l.strip()]
print(f'Loaded {len(SAMPLES)} samples')

## Install

In [ ]:
!git clone https://github.com/gangweix/pixel-perfect-depth /content/pixel-perfect-depth 2>/dev/null || true
!pip install -q timm einops xformers
!grep -v '^torch==' /content/pixel-perfect-depth/requirements.txt | pip install -q -r /dev/stdin

## Run

In [ ]:
MODEL_NAME = 'pixel_perfect'
raw_dir = '/content/ppd_raw'
os.makedirs(raw_dir, exist_ok=True)

times, failed = [], 0
for img_name in SAMPLES:
    stem = img_name.split('.')[0]
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, 'run.py',
         '--img_path', os.path.join(DATASET_IMAGES, img_name),
         '--outdir', raw_dir, '--save_npy'],
        cwd='/content/pixel-perfect-depth',
        capture_output=True, text=True, timeout=300
    )
    times.append(time.time()-t0)
    if r.returncode != 0:
        failed += 1
        if failed <= 3: print(f'WARN {img_name}:', r.stderr[-200:].strip())
        continue
    for fname in [f'{stem}_depth.npy', f'{stem}.npy', f'{stem}_pred.npy', f'{stem}_pred_depth.npy']:
        fp = os.path.join(raw_dir, fname)
        if os.path.exists(fp):
            save_depth(np.load(fp), stem, MODEL_NAME)
            break

print(f'Done -- {len(times)-failed}/{len(SAMPLES)} ok, avg {np.mean(times):.2f}s/img')
clear_gpu()

## Confirm saved

In [ ]:
model_dir = os.path.join(SAVE_DIR, MODEL_NAME)
n_depth = len(os.listdir(os.path.join(model_dir, 'depth')))
n_vis   = len(os.listdir(os.path.join(model_dir, 'vis')))
print(f'{MODEL_NAME}: {n_depth} depth maps, {n_vis} visualizations saved to Drive')